## Scenario 1: A single data scientist participating in an ML competition

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

In [1]:
import mlflow

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns'


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns/932244146200333313', creation_time=1753918365062, experiment_id='932244146200333313', last_update_time=1753918365062, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns/0', creation_time=1753918301650, experiment_id='0', last_update_time=1753918301650, lifecycle_stage='active', name='Default', tags={}>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name="models", input_example=X[:5])
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

default artifacts URI: 'file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns/932244146200333313/2476cc11ca8c4e229cd3f99cdb8b93fd/artifacts'


2025/07/30 23:35:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.

모델을 MLflow에 저장할 때 **입력 예시(input_example)**와 signature를 지정하지 않았다
추가하면 MLflow가 자동으로 모델의 입력/출력 타입(signature)을 추론해 기록합니다.

mlflow.sklearn.log_model(lr, name="models", input_example=X[:5])


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns/932244146200333313', creation_time=1753918365062, experiment_id='932244146200333313', last_update_time=1753918365062, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///workspaces/mlops-zoomcamp/02-Experiment%20Tracking/examples/mlruns/0', creation_time=1753918301650, experiment_id='0', last_update_time=1753918301650, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [7]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException: # MLflow 3.x 이상 버전에서는 예외가 발생하지 않는 듯하다
    print("It's not possible to access the model registry :(")

be store로 로컬 파일시스템을 사용할 때는 기본적으로 모델 레지스트리 기능이 비활성화되어 있다. 모델 레지스트리는 MLflow Tracking Server를 DB(예: SQLite, MySQL, PostgreSQL)와 연동해야 활성화된다.
